In [1]:
import pandas as pd

In [2]:
business_df = pd.read_json('dataset/yelp_dataset/yelp_academic_dataset_business.json', lines=True)
len(business_df)

150346

In [3]:
# Cleaning yelp_academic_dataset_business.json file
# Filtered file to only show businesses with more than 20 reviews

business_df = business_df[business_df['review_count'] >=20]
len(business_df)

61919

In [4]:
business_df.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='object')

In [5]:
# Testing to see if there is a correlation between avg ratings
# and what state the restraunt is in

state_average = business_df.groupby('state')['stars'].mean()
print(state_average)

state
AB    3.591451
AZ    3.582670
CA    3.916926
DE    3.416164
FL    3.636411
ID    3.670187
IL    3.449919
IN    3.635359
LA    3.750108
MA    1.500000
MO    3.575114
NC    2.000000
NJ    3.484557
NV    3.687778
PA    3.595802
SD    4.500000
TN    3.580496
Name: stars, dtype: float64


In [6]:
# Removed unnecessary columns
business_df.drop(columns=['address', 'city', 'postal_code', 'latitude', 'longitude', 'is_open', 'hours', 'attributes', 'categories', 'name'], inplace=True)
business_df.columns

Index(['business_id', 'state', 'stars', 'review_count'], dtype='object')

In [ ]:
# Renaming the 'stars' feature to 'business_star'
business_df.rename(columns={'stars': 'business_star'}, inplace=True)
business_df.columns

Index(['business_id', 'state', 'business_star', 'review_count'], dtype='object')

In [ ]:
# The file was too big for my laptop to handle
# Loaded the files in chunks, dropped unnecessary 
# columns and wrote it out to memory

file_in = 'dataset/yelp_dataset/yelp_academic_dataset_review.json'
file_out = 'dataset/cleaned_dataset/cleaned_review.json'

chunk_size = 200000

with open(file_out, 'w') as out:
    first_chunk = True

    for chunk in pd.read_json(file_in, lines=True, chunksize=chunk_size):
        chunk.drop(columns=['user_id', 'useful', 'funny', 'cool', 'date'], inplace=True)
        chunk.dropna(inplace=True)

        if first_chunk:
            chunk.to_json(out, orient='records', lines=True)
        else:
            chunk.to_json(out, orient='records', lines=True, mode='a')

In [9]:
review_df = pd.read_json('dataset/cleaned_dataset/cleaned_review.json', lines=True)
review_df.columns

Index(['review_id', 'business_id', 'stars', 'text'], dtype='object')

In [10]:
business_df.columns

Index(['business_id', 'state', 'business_star', 'review_count'], dtype='object')

In [ ]:
# finding the average of all review ratings and merging it to bussiness

average_review_ratings_df = review_df[['business_id', 'stars']]
average_review_ratings_df = average_review_ratings_df.groupby('business_id', as_index=False).agg(average_review_ratings_df=('stars', 'mean'))

average_review_ratings_df.head()

,business_id,average_review_ratings_df
0,---kPU91CF4Lq2-WlRu9Lw,4.500000
1,--0iUa4sNDFiZFrAdIWhZQ,3.214286
2,--30_8IhuyMHbSOcNWd6DQ,3.555556
3,--7PUidqRWpRSpXebiyxTg,1.750000
4,--7jw19RH9JKXgFohspgQw,4.230769


In [12]:
business_df = business_df.merge(average_review_ratings_df, on='business_id', how='inner')
business_df.head

<bound method NDFrame.head of                   business_id state  business_star  review_count  \
0      tUFrWirKiKi_TAnsVWINQQ    AZ            3.5            22   
1      MTSW4McQd7CbVtyjqoe9mw    PA            4.0            80   
2      il_Ro8jwPlHresjw9EGmBg    IN            2.5            28   
3      0bPLkL0QhhPO5kt1_EXmNQ    FL            4.5           100   
4      MUTTqe8uqyMdBl186RmNeA    PA            4.0           245   
...                       ...   ...            ...           ...   
61914  GeEveoOaU2YKD7jJtEfA_g    NV            5.0            34   
61915  qQ7FHvkGEMqoPKKXPk4gjA    AZ            2.5            67   
61916  LJ4GjQ1HL6kqvIPpNUNNaQ    PA            4.5            39   
61917  WnT9NIzQgLlILjPT0kEcsQ    PA            4.5            35   
61918  mtGm22y5c2UHNXDFAjaPNw    IL            4.0            24   

       average_review_ratings_df  
0                       3.500000  
1                       4.057471  
2                       2.413793  
3            

In [13]:
review_df.columns

Index(['review_id', 'business_id', 'stars', 'text'], dtype='object')

In [14]:
review_df

,review_id,business_id,stars,text
0,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,3,"If you decide to eat here, just be aware it is..."
1,BiTunyQ73aT9WBnpR9DZGw,7ATYjTIgM3jUlt4UM3IypQ,5,I've taken a lot of spin classes over the year...
2,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,3,Family diner. Had the buffet. Eclectic assortm...
3,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,5,"Wow! Yummy, different, delicious. Our favo..."
4,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,4,Cute interior and owner (?) gave us tour of up...
...,...,...,...,...
6990275,H0RIamZu0B0Ei0P4aeh3sQ,jals67o91gcrD4DC81Vk6w,5,Latest addition to services from ICCU is Apple...
6990276,shTPgbgdwTHSuU67mGCmZQ,2vLksaMmSEcGbjI5gywpZA,5,"This spot offers a great, affordable east week..."
6990277,YNfNhgZlaaCO5Q_YJR4rEw,R1khUUxidqfaJmcpmGd4aw,4,This Home Depot won me over when I needed to g...
6990278,i-I4ZOhoX70Nw5H0FwrQUA,Rr9kKArrMhSLVE9a53q-aA,5,For when I'm feeling like ignoring my calorie-...


In [ ]:
# Dropping the review_id and y_label

review_df.drop(columns=['review_id', 'stars'], inplace=True)
review_df.columns


Index(['business_id', 'text'], dtype='object')

In [ ]:
# concatenating all the reviews onto one column and merging
# it with the main df

review_df = review_df.groupby('business_id', as_index=False).agg(text=('text', '\n'.join))
review_df.head

<bound method NDFrame.head of                    business_id  \
0       ---kPU91CF4Lq2-WlRu9Lw   
1       --0iUa4sNDFiZFrAdIWhZQ   
2       --30_8IhuyMHbSOcNWd6DQ   
3       --7PUidqRWpRSpXebiyxTg   
4       --7jw19RH9JKXgFohspgQw   
...                        ...   
150341  zznZqH9CiAznbkV6fXyHWA   
150342  zztOG2cKm87I6Iw_tleZsQ   
150343  zzu6_r3DxBJuXcjnOYVdTw   
150344  zzw66H6hVjXQEt0Js3Mo4A   
150345  zzyx5x0Z7xXWWvWnZFuxlQ   

                                                     text  
0       Ate here for the 1st time on Saturday 08/07/20...  
1       Very good San Salvadorian place ! Authentic an...  
2       We stopped going to Action Karate in December ...  
3       This place is disgusting, and proof that Edmon...  
4       This is the best dentist in the area, hands do...  
...                                                   ...  
150341  Oh my gosh! Yum yum yum! I saw Que Pasta start...  
150342  This course is exactly what I needed to get me...  
150343  Probably the 

In [17]:
business_df = business_df.merge(review_df, on='business_id', how='inner')
business_df.columns

Index(['business_id', 'state', 'business_star', 'review_count',
       'average_review_ratings_df', 'text'],
      dtype='object')

In [18]:
business_df.head()

,business_id,state,business_star,review_count,average_review_ratings_df,text
0,tUFrWirKiKi_TAnsVWINQQ,AZ,3.5,22,3.500000,We are fans of Target. They seem to have a li...
1,MTSW4McQd7CbVtyjqoe9mw,PA,4.0,80,4.057471,This is nice little Chinese bakery in the hear...
2,il_Ro8jwPlHresjw9EGmBg,IN,2.5,28,2.413793,Went there at 4am and there was only one waitr...
3,0bPLkL0QhhPO5kt1_EXmNQ,FL,4.5,100,4.386792,The worst Chicken Parm. Sandwich I've ever eat...
4,MUTTqe8uqyMdBl186RmNeA,PA,4.0,245,4.200000,Stopped in to check out this new spot around t...


In [19]:
business_df.shape

(61919, 6)

In [ ]:
# Renaming feature columns

business_df.rename({'average_review_ratings_df': 'avg_review_ratings', 'text': 'reviews'}, inplace=True)
business_df.columns

Index(['business_id', 'state', 'business_star', 'review_count',
       'average_review_ratings_df', 'text'],
      dtype='object')

In [21]:
business_df.head

<bound method NDFrame.head of                   business_id state  business_star  review_count  \
0      tUFrWirKiKi_TAnsVWINQQ    AZ            3.5            22   
1      MTSW4McQd7CbVtyjqoe9mw    PA            4.0            80   
2      il_Ro8jwPlHresjw9EGmBg    IN            2.5            28   
3      0bPLkL0QhhPO5kt1_EXmNQ    FL            4.5           100   
4      MUTTqe8uqyMdBl186RmNeA    PA            4.0           245   
...                       ...   ...            ...           ...   
61914  GeEveoOaU2YKD7jJtEfA_g    NV            5.0            34   
61915  qQ7FHvkGEMqoPKKXPk4gjA    AZ            2.5            67   
61916  LJ4GjQ1HL6kqvIPpNUNNaQ    PA            4.5            39   
61917  WnT9NIzQgLlILjPT0kEcsQ    PA            4.5            35   
61918  mtGm22y5c2UHNXDFAjaPNw    IL            4.0            24   

       average_review_ratings_df  \
0                       3.500000   
1                       4.057471   
2                       2.413793   
3        

In [ ]:
# writing out the main dataframe to memory

chunk_size = 200000
file_out = 'dataset/cleaned_dataset/training_dataset.json'

with open(file_out, 'w') as out:
    for i in range(0, len(business_df), chunk_size):
        chunk = business_df.iloc[i:i+chunk_size]
        chunk.to_json(out, orient='records', lines=True)
